# Thực nghiệm thu nhỏ KB-VQA trên OK-VQA
So sánh **không RAG** và **RAG đơn giản** (entity → tìm Wikipedia → rerank → trả lời), lấy cảm hứng từ ý tưởng Processing–Retrieval–Filtering của Wiki-PRF.

- Mô hình: Qwen2.5-VL-3B-Instruct (không huấn luyện thêm, chỉ suy luận)
- Dữ liệu: tập con OK-VQA (val)
- Chạy trên Colab: **Runtime > Change runtime type > T4 GPU**

Lưu ý: đây là thiết lập thu nhỏ, không so trực tiếp với số liệu trong bài báo.

In [1]:
!pip -q install -U transformers accelerate qwen-vl-utils datasets sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import torch, re, json, time, requests
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from sentence_transformers import SentenceTransformer, util

# ===== Cấu hình =====
N_SAMPLES = 200      # số mẫu OK-VQA (càng nhiều càng ổn định)
TOP_K     = 2        # số đoạn Wikipedia giữ lại sau khi rerank
N_SEARCH  = 5        # số bài Wikipedia lấy về cho mỗi truy vấn
CTX_CHARS = 800      # cắt mỗi đoạn tri thức tối đa bấy nhiêu ký tự
MODEL_ID  = "Qwen/Qwen2.5-VL-3B-Instruct"


In [3]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=128*28*28, max_pixels=512*28*28)  # giới hạn kích thước ảnh cho đỡ tốn VRAM
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

@torch.no_grad()
def ask(image, prompt, max_new_tokens=32):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(messages)
    inputs = processor(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    out = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

... (15 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# Dữ liệu OK-VQA (tập con của val2014)
ds = load_dataset("lmms-lab/OK-VQA", split=f"val2014[:{N_SAMPLES}]")
print(ds)
print(ds[0]["question"], ds[0]["answers"])

README.md:   0%|          | 0.00/488 [00:00<?, ?B/s]

data/val2014-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  418MB            

data/val2014-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/val2014-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  413MB            

data/val2014-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating val2014 split:   0%|          | 0/5046 [00:00<?, ? examples/s]

Dataset({
    features: ['question_id', 'image', 'question', 'answers', 'question_type', 'answer_type'],
    num_rows: 200
})
What sport can you use this for? ['racing', 'racing', 'racing', 'racing', 'racing', 'racing', 'motocross', 'motocross', 'riding', 'riding']


## Các bước của pipeline RAG

In [5]:
# --- Processing: nhận diện thực thể chính trong ảnh (giống công cụ captioning) ---
def get_entity(image, question=None):
    # Không đưa câu hỏi vào để mô hình không tự "trả lời hộ" trong bước nhận diện thực thể
    p = ("Name the main object, landmark, animal, plant, food or brand in this image "
         "using a short noun phrase of 1-4 words. Output only the name.")
    return ask(image, p, max_new_tokens=12).strip(" .\n")

# --- Retrieval: tìm bài Wikipedia qua API ---
HEADERS = {"User-Agent": "KBVQA-student-project/0.1 (ducvan0505@gmail.com)"}
wiki_stats = {"ok": 0, "fail": 0}

def wiki_search(query, n=N_SEARCH, retries=3):
    params = {"action": "query", "format": "json", "generator": "search",
              "gsrsearch": query, "gsrlimit": n, "prop": "extracts",
              "exintro": 1, "explaintext": 1, "exlimit": "max"}
    for attempt in range(retries):
        try:
            r = requests.get("https://en.wikipedia.org/w/api.php",
                             params=params, headers=HEADERS, timeout=20)
            if r.status_code != 200:
                if attempt == 0: print("HTTP", r.status_code, r.text[:150])
                time.sleep(2 * (attempt + 1)); continue
            pages = r.json().get("query", {}).get("pages", {})
            wiki_stats["ok"] += 1
            time.sleep(0.3)   # giãn cách để tránh bị giới hạn tốc độ
            return [(p["title"], p.get("extract", "")[:CTX_CHARS])
                    for p in pages.values() if p.get("extract")]
        except Exception as e:
            time.sleep(2 * (attempt + 1))
    wiki_stats["fail"] += 1
    return []

# --- Filtering: xếp hạng lại theo độ giống với (câu hỏi + thực thể), giữ top-k ---
def rerank(query, docs, k=TOP_K):
    if not docs: return []
    q = embedder.encode(query, convert_to_tensor=True)
    d = embedder.encode([f"{t}. {x}" for t, x in docs], convert_to_tensor=True)
    scores = util.cos_sim(q, d)[0]
    idx = scores.argsort(descending=True)[:k].tolist()
    return [docs[i] for i in idx]

STYLE = ("Answer with the shortest possible answer: ideally ONE common word (at most 3 words), "
         "no full sentence, no punctuation, lowercase. Prefer the simple everyday word "
         "(e.g. 'cowboy' not 'cowboy hat', 'cloth' not 'serge').")
BASE_PROMPT = STYLE + "\nQuestion: {q}\nAnswer:"
RAG_PROMPT  = ("Background knowledge (may be partly irrelevant; ignore it if it does not help, "
               "and rely mainly on the image):\n{ctx}\n\n" + STYLE + "\nQuestion: {q}\nAnswer:")


## Đánh giá (VQA accuracy mềm)

In [6]:
def norm(s):
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def vqa_acc(pred, gts):
    # VQA accuracy chuẩn (khớp chính xác sau chuẩn hóa)
    gts = [g["answer"] if isinstance(g, dict) else g for g in gts]
    p = norm(pred)
    return min(1.0, sum(norm(g) == p for g in gts) / 3.0)

def stem(w):
    return w[:-1] if len(w) > 3 and w.endswith("s") else w
def lenient_match(p, g):
    p, g = [stem(w) for w in norm(p).split()], [stem(w) for w in norm(g).split()]
    if not p or not g: return False
    # khớp nếu một cụm chứa trọn cụm kia (vd. "cowboy hat" ~ "cowboy", "horns" ~ "horn")
    return set(p) <= set(g) or set(g) <= set(p)

def vqa_acc_lenient(pred, gts):
    # Phiên bản nới lỏng: chấp nhận khớp một phần theo từ
    gts = [g["answer"] if isinstance(g, dict) else g for g in gts]
    return min(1.0, sum(lenient_match(pred, g) for g in gts) / 3.0)


In [7]:
rows = []
for ex in tqdm(ds):
    img, q, gts = ex["image"].convert("RGB"), ex["question"], ex["answers"]
    # 1) baseline: không dùng tri thức ngoài
    a0 = ask(img, BASE_PROMPT.format(q=q))
    # 2) RAG
    ent  = get_entity(img, q)
    docs = rerank(f"{q} {ent}", wiki_search(ent))
    ctx  = "\n".join(f"[{t}] {x}" for t, x in docs)
    a1 = ask(img, RAG_PROMPT.format(ctx=ctx, q=q)) if docs else a0
    rows.append(dict(question=q, gt=[g["answer"] if isinstance(g, dict) else g for g in gts][:3],
                     entity=ent, titles=[t for t, _ in docs],
                     pred_base=a0, pred_rag=a1, retrieved=bool(docs),
                     acc_base=vqa_acc(a0, gts), acc_rag=vqa_acc(a1, gts),
                     len_base=vqa_acc_lenient(a0, gts), len_rag=vqa_acc_lenient(a1, gts)))
df = pd.DataFrame(rows)
print(f"[Chuẩn]   Không RAG: {df.acc_base.mean()*100:.1f}% | Có RAG: {df.acc_rag.mean()*100:.1f}%")
print(f"[Nới lỏng] Không RAG: {df.len_base.mean()*100:.1f}% | Có RAG: {df.len_rag.mean()*100:.1f}%")
print("Thống kê truy vấn Wikipedia:", wiki_stats)
sub = df[df.retrieved]
print(f"Truy xuất thành công: {len(sub)}/{len(df)} mẫu")
if len(sub):
    print(f"  Trên các mẫu này -> Không RAG: {sub.acc_base.mean()*100:.1f}% | Có RAG: {sub.acc_rag.mean()*100:.1f}% (chuẩn)")


  0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


[Chuẩn]   Không RAG: 54.5% | Có RAG: 49.5%
[Nới lỏng] Không RAG: 61.0% | Có RAG: 58.3%
Thống kê truy vấn Wikipedia: {'ok': 200, 'fail': 0}
Truy xuất thành công: 200/200 mẫu
  Trên các mẫu này -> Không RAG: 54.5% | Có RAG: 49.5% (chuẩn)


Em làm bản thu nhỏ vì máy không chạy nổi Wiki-PRF gốc: dùng Qwen2.5-VL-3B, 200 mẫu OK-VQA, giữ 3 bước tách thực thể, tìm Wikipedia, lọc lại. Kết quả là RAG đơn giản không cải thiện mà còn giảm từ 55,0% xuống 50,0%. Em nghĩ do mô hình 3B nhỏ nên dễ bị nhiễu bởi đoạn Wikipedia không sát câu hỏi, và em chỉ lọc đơn giản bằng MiniLM. Bài báo gốc có thêm bước huấn luyện để mô hình biết lọc thông tin, còn bản của em chưa có

In [8]:
# Mẫu RAG giúp được / làm hỏng
display(df[df.acc_rag > df.acc_base][["question","gt","entity","titles","pred_base","pred_rag"]].head(8))
display(df[df.acc_rag < df.acc_base][["question","gt","entity","titles","pred_base","pred_rag"]].head(8))
print("RAG tốt hơn:", (df.acc_rag > df.acc_base).sum(), "| kém hơn:", (df.acc_rag < df.acc_base).sum(),
      "| bằng nhau:", (df.acc_rag == df.acc_base).sum())
df.to_csv("okvqa_rag_results.csv", index=False)

,question,gt,entity,titles,pred_base,pred_rag
12,What is the name of the items the hot dog are ...,"[condiments, condiments, condiments]",hot dog,"[Hot dog, Hot dog variations]",pickles,condiments
15,What type of plane is that?,"[commercial, commercial, commercial]",TAS Spirit of Australia,"[Spirit of Tasmania, Australia]",tas,passenger
43,What type of bird is this?,"[blue jay, blue jay, blue jay]",Blue bird,"[Bluebird, The Blue Bird (play)]",blue and yellow,bluebird
44,How can you tell that the people are not upset?,"[they are smiling, they are smiling, they are ...",Pizza,"[Pizza (disambiguation), Pizza]",they're smiling,smiling
82,What is this truck moving?,"[furniture, furniture, furniture]",truck,"[Truck (disambiguation), Truck]",recalled items,cargo
87,What fills the balloons?,"[helium, helium, helium]",birthday cake,"[Birthday cake, The Birthday Cake]",air,helium
102,What is the white cloud behind the jet called?,"[contrail, contrail, contrail]",jet plane,"[Jet aircraft, Jay Jay the Jet Plane]",sonic boom,contrail
109,What is the name of this dish?,"[pancakes, pancakes, pancakes]",pancakes,"[Crispy pancakes, Pancake]",pancake,pancakes


,question,gt,entity,titles,pred_base,pred_rag
28,What flavor is this pastry?,"[blueberry, blueberry, blueberry]",cake,"[Cake, Cake (disambiguation)]",chocolate,vanilla
33,How many watts is that microwave?,"[800, 800, 800]",umbrella,"[Umbrella (song), Umbrella organization]",1000,500
35,What south american country usually has this c...,"[brazil, brazil, brazil]",cow,"[Cattle, Cowes]",brazil,paraguay
45,How fast is the vehicle pictured capable of re...,"[80 mph, 80 mph, 80 mph]",train,"[A Train, The Train]",100 mph,slow
47,What sates are these grown in?,"[florida california, florida california, flori...",orange,"[Orange (fruit), Shades of orange]",california,tropical
68,What character does this clock look like?,"[mickey mouse, mickey mouse, mickey mouse]",Mickey Mouse clock,"[Clock Cleaners, Mickey Mouse]",mickey mouse,mickey
69,What fruit comes from these trees?,"[coconut, coconut, coconut]",clock tower,"[Clock Tower, Clock tower]",coconut,palm
74,How tall do these animals get?,"[20 feet, 20 feet, 20 feet]",giraffe,"[Giraffe, Reticulated giraffe]",20 feet,12 meters


RAG tốt hơn: 13 | kém hơn: 23 | bằng nhau: 164


## Ablation nhỏ (tùy chọn)
Đổi `TOP_K` (1, 2, 3) hoặc `N_SEARCH`, hoặc thay `wiki_search(ent)` bằng `wiki_search(q)` (truy vấn bằng câu hỏi thay vì thực thể) rồi chạy lại ô đánh giá để so sánh.

## Kiểm tra độ vững (chạy sau ô đánh giá chính)
Các ô dưới đây dùng lại `df` để kiểm định ý nghĩa thống kê, rồi chạy thêm ablation và một tập mẫu ngẫu nhiên khác. Điền kết quả vào tiểu luận sau khi chạy.

In [9]:
# 1) Kiểm định ý nghĩa: sign test + bootstrap cho hiệu RAG - không RAG
import numpy as np
from scipy.stats import binomtest
d = (df.acc_rag - df.acc_base).values
better, worse = int((d > 0).sum()), int((d < 0).sum())
print("tốt hơn:", better, "| kém hơn:", worse, "| p (sign test, 2 phía) =", round(binomtest(better, better+worse, 0.5).pvalue, 3))
rng = np.random.default_rng(0)
boot = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(10000)]
print(f"Hiệu trung bình = {d.mean()*100:.1f} điểm, KTC 95% bootstrap = [{np.percentile(boot,2.5)*100:.1f}, {np.percentile(boot,97.5)*100:.1f}]")

tốt hơn: 13 | kém hơn: 23 | p (sign test, 2 phía) = 0.132
Hiệu trung bình = -5.0 điểm, KTC 95% bootstrap = [-10.2, 0.0]


In [10]:
# 2) Phân loại lỗi: in các mẫu RAG kém hơn/tốt hơn để gán nhãn tay (thực thể sai / tri thức chung chung / khác)
pd.set_option("display.max_colwidth", 60)
display(df[df.acc_rag < df.acc_base][["question","gt","entity","titles","pred_base","pred_rag"]])
display(df[df.acc_rag > df.acc_base][["question","gt","entity","titles","pred_base","pred_rag"]])

,question,gt,entity,titles,pred_base,pred_rag
28,What flavor is this pastry?,"[blueberry, blueberry, blueberry]",cake,"[Cake, Cake (disambiguation)]",chocolate,vanilla
33,How many watts is that microwave?,"[800, 800, 800]",umbrella,"[Umbrella (song), Umbrella organization]",1000,500
35,What south american country usually has this climate?,"[brazil, brazil, brazil]",cow,"[Cattle, Cowes]",brazil,paraguay
45,How fast is the vehicle pictured capable of reaching?,"[80 mph, 80 mph, 80 mph]",train,"[A Train, The Train]",100 mph,slow
47,What sates are these grown in?,"[florida california, florida california, florida califor...",orange,"[Orange (fruit), Shades of orange]",california,tropical
68,What character does this clock look like?,"[mickey mouse, mickey mouse, mickey mouse]",Mickey Mouse clock,"[Clock Cleaners, Mickey Mouse]",mickey mouse,mickey
69,What fruit comes from these trees?,"[coconut, coconut, coconut]",clock tower,"[Clock Tower, Clock tower]",coconut,palm
74,How tall do these animals get?,"[20 feet, 20 feet, 20 feet]",giraffe,"[Giraffe, Reticulated giraffe]",20 feet,12 meters
77,Why so many green vegetables?,"[healthy, healthy, healthy]",vegetables,"[Leaf vegetable, List of vegetables]",healthy,good for you
85,What is a type of horseback sport?,"[polo, polo, polo]",horse,"[Equestrianism, Horse racing]",polo,horse racing


,question,gt,entity,titles,pred_base,pred_rag
12,What is the name of the items the hot dog are topped with?,"[condiments, condiments, condiments]",hot dog,"[Hot dog, Hot dog variations]",pickles,condiments
15,What type of plane is that?,"[commercial, commercial, commercial]",TAS Spirit of Australia,"[Spirit of Tasmania, Australia]",tas,passenger
43,What type of bird is this?,"[blue jay, blue jay, blue jay]",Blue bird,"[Bluebird, The Blue Bird (play)]",blue and yellow,bluebird
44,How can you tell that the people are not upset?,"[they are smiling, they are smiling, they are smiling]",Pizza,"[Pizza (disambiguation), Pizza]",they're smiling,smiling
82,What is this truck moving?,"[furniture, furniture, furniture]",truck,"[Truck (disambiguation), Truck]",recalled items,cargo
87,What fills the balloons?,"[helium, helium, helium]",birthday cake,"[Birthday cake, The Birthday Cake]",air,helium
102,What is the white cloud behind the jet called?,"[contrail, contrail, contrail]",jet plane,"[Jet aircraft, Jay Jay the Jet Plane]",sonic boom,contrail
109,What is the name of this dish?,"[pancakes, pancakes, pancakes]",pancakes,"[Crispy pancakes, Pancake]",pancake,pancakes
117,How long does this animal usually live?,"[10 years, 10 years, 10 years]",sheep,"[Sheep, Bighorn sheep]",12 years,10 years
132,What kind of park is this picture taken in?,"[skatepark, skatepark, skatepark]",skateboarder,"[Skateboard, Skateboarding]",skate,skatepark


In [11]:
# 3) Ablation + tập mẫu ngẫu nhiên khác (không trùng 200 mẫu đầu). Dùng cùng prompt để so sánh công bằng.
import random
def run_eval(dataset, query_mode="entity", top_k=TOP_K):
    res = []
    for ex in tqdm(dataset):
        img, q, gts = ex["image"].convert("RGB"), ex["question"], ex["answers"]
        a0 = ask(img, BASE_PROMPT.format(q=q))
        ent = get_entity(img)
        docs = wiki_search(ent if query_mode == "entity" else (q if query_mode == "question" else f"{ent} {q}"))
        docs = rerank(f"{q} {ent}", docs, k=top_k)
        ctx = "\n".join(f"[{t}] {x}" for t, x in docs)
        a1 = ask(img, RAG_PROMPT.format(ctx=ctx, q=q)) if docs else a0
        res.append((vqa_acc(a0, gts), vqa_acc(a1, gts)))
    r = np.array(res); return r[:,0].mean()*100, r[:,1].mean()*100

full = load_dataset("lmms-lab/OK-VQA", split="val2014")
idx = random.Random(42).sample(range(N_SAMPLES, len(full)), 300)
rand_ds = full.select(idx)
for mode in ["entity", "entity+question", "question"]:
    print(mode, "-> (không RAG, có RAG) =", run_eval(rand_ds, mode))
for k in (1, 3):
    print("top_k =", k, "->", run_eval(rand_ds, "entity", k))

  0%|          | 0/300 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


entity -> (không RAG, có RAG) = (np.float64(60.66666666666667), np.float64(57.22222222222222))


  0%|          | 0/300 [00:00<?, ?it/s]

entity+question -> (không RAG, có RAG) = (np.float64(60.66666666666667), np.float64(56.44444444444444))


  0%|          | 0/300 [00:00<?, ?it/s]

question -> (không RAG, có RAG) = (np.float64(60.66666666666667), np.float64(49.888888888888886))


  0%|          | 0/300 [00:00<?, ?it/s]

top_k = 1 -> (np.float64(60.66666666666667), np.float64(59.55555555555555))


  0%|          | 0/300 [00:00<?, ?it/s]

top_k = 3 -> (np.float64(60.66666666666667), np.float64(57.1111111111111))
